---

## Paso 1: Cargar y validar la calidad de los datos

---



In [1]:
# importar librerías
import pandas as pd
from statsmodels.stats.proportion import proportions_ztest

In [2]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [3]:
# explorar datasets
orders.info()
orders.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [4]:
conteo=orders['id_pedido'].value_counts()
repetido= conteo[conteo>=2]
orders[orders['id_pedido'].isin(repetido.index)]

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
734,order_734,user_6347,2025-05-02,Colombia,desktop,organic,Blender-XL-Red,Hogar,1.0,58.00,0.0,58.00
812,order_812,user_1530,2025-03-31,Argentina,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,167.32,10.0,157.32
974,order_974,user_3262,2025-05-04,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,2.0,268.58,5.0,532.16
1361,order_1361,user_507,2025-06-27,Mexico,desktop,paid_search,Laptop-Gaming-16GB,Electronica,2.0,239.61,0.0,479.22
1735,order_1735,user_728,2025-05-13,Mexico,mobile,social,Tablet-Standard-64GB,Electronica,2.0,419.04,0.0,838.08
...,...,...,...,...,...,...,...,...,...,...,...,...
25095,order_3913,user_380,2025-02-18,Argentina,desktop,paid_search,Phone-Pro-128GB,Electronica,1.0,82.28,0.0,82.28
25096,order_23405,user_7833,2025-04-04,Colombia,mobile,paid_search,Phone-Pro-128GB,Electronica,2.0,99.25,5.0,193.50
25097,order_5615,user_5417,2025-05-13,Colombia,desktop,social,Blender-XL-Red,Hogar,2.0,450.35,5.0,895.69
25098,order_812,user_1530,2025-03-31,Argentina,desktop,organic,Tablet-Standard-64GB,Electronica,1.0,167.32,10.0,157.32


In [5]:
negativos = orders[orders['cantidad']<0]
negativos.head(10)

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
266,order_266,user_7011,2025-03-13,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-2.0,101.31,10.0,-192.62
267,order_267,user_1087,2025-05-07,NaN,desktop,social,Phone-Pro-128GB,Electronica,-1.0,43.50,5.0,-38.50
268,order_268,user_84,2025-02-19,NaN,desktop,organic,Phone-Pro-128GB,Electronica,-1.0,497.65,5.0,-492.65
269,order_269,user_3323,2025-05-25,NaN,desktop,paid_search,Phone-Pro-128GB,Electronica,-1.0,423.53,0.0,-423.53


In [6]:
catalog.info()
catalog.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000


In [7]:
marketing.info()
marketing.describe()
marketing.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


---

### Revisión y calidad de datos

In [8]:
orders['fecha_hora_pedido']=pd.to_datetime(orders['fecha_hora_pedido'])
orders['fecha_hora_pedido'].isna().value_counts()

False    25100
Name: fecha_hora_pedido, dtype: int64

In [9]:
orders = orders.dropna(subset=['fuente_referencia','cantidad','dispositivo']).reset_index(drop=True)
orders = orders[orders['cantidad']>=0]
orders['pais'] = orders ['pais'].fillna('Desconocido')
orders = orders.drop_duplicates(subset=['id_pedido'])
orders.info()
orders.describe()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 24896 entries, 0 to 24899
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   id_pedido           24896 non-null  object        
 1   id_usuario          24896 non-null  object        
 2   fecha_hora_pedido   24896 non-null  datetime64[ns]
 3   pais                24896 non-null  object        
 4   dispositivo         24896 non-null  object        
 5   fuente_referencia   24896 non-null  object        
 6   nombre_producto     24896 non-null  object        
 7   categoria_producto  24896 non-null  object        
 8   cantidad            24896 non-null  float64       
 9   precio_unitario     24896 non-null  float64       
 10  monto_descuento     24896 non-null  float64       
 11  monto_total         24896 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(7)
memory usage: 2.5+ MB


,cantidad,precio_unitario,monto_descuento,monto_total
count,24896.000000,24896.000000,24896.000000,2.489600e+04
mean,7.127852,259.382219,4.502531,2.086615e+03
std,297.191630,138.660727,5.224288,9.935442e+04
min,1.000000,20.030000,0.000000,5.240000e+00
25%,1.000000,138.570000,0.000000,1.807175e+02
50%,2.000000,258.795000,0.000000,3.417500e+02
75%,2.000000,380.297500,10.000000,5.184825e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [10]:
marketing["canal"] = marketing["id_campaña"].str.split("_").str[0]
marketing['fecha']=pd.to_datetime(marketing['fecha'])
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   fecha       1620 non-null   datetime64[ns]
 1   pais        1620 non-null   object        
 2   id_campaña  1620 non-null   object        
 3   canal       1620 non-null   object        
 4   gasto       1620 non-null   float64       
dtypes: datetime64[ns](1), float64(1), object(3)
memory usage: 63.4+ KB


In [11]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

Los datos limpios fueron exportados, de la tabla orders se eliminaron filas con las que contaban con valores nulos en las columnas fuente referencia, cantidad y dispositivo, ya que las mismas filas de pequeno numero sobre el total de datos compartian estas tres columnas vacias. Se eliminaron filas con cantidades negativas, las cuales podrian representar devoluciones en la aplicacion, que podian afectar los calculos finales y estadisticas, ademas se eliminaron filas duplicadas a las cuales se les verifico que fueran exactamente iguales.

---

## Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

In [12]:
revenue = orders['monto_total'].sum()
print(f'Revenue = ${revenue:.2f}')
orders_with_cost = orders.merge(catalog[['nombre_producto', 'costo_unitario']], on='nombre_producto', how='left')
orders_with_cost['costo_total'] = orders_with_cost['cantidad'] * orders_with_cost['costo_unitario']
costo_total = orders_with_cost['costo_total'].sum()
print(f'Costo Total = ${costo_total:.2f}')
gasto_marketing = marketing['gasto'].sum()
print(f'Gasto Total en Marketing = ${gasto_marketing:.2f}')
ganancia_neta = revenue - costo_total - gasto_marketing
print(f'Ganancia Neta = ${ganancia_neta:.2f}')
margen = 100*(ganancia_neta/revenue)
print(f'Margen de Ganancia = {margen:.2f}%')

Revenue = $51948363.19
Costo Total = $43122388.54
Gasto Total en Marketing = $2871843.53
Ganancia Neta = $5954131.12
Margen de Ganancia = 11.46%


In [13]:
tiquete_promedio = orders['monto_total'].mean()
cantidad_promedio = orders['cantidad'].mean()
producto_mas_vendido = orders['nombre_producto'].value_counts().idxmax()
gasto_marketing = marketing.groupby('canal')['gasto'].sum()
print(f'Tiquete Promedio ={tiquete_promedio:.2f}')
print(f'Cantidad Promedio ={cantidad_promedio:.2f}')
print(f'Producto mas Vendido ={producto_mas_vendido}')
print(f'Gasto de Marketing por Canal ={gasto_marketing}')

Tiquete Promedio =2086.61
Cantidad Promedio =7.13
Producto mas Vendido =Blender-XL-Red
Gasto de Marketing por Canal =canal
organic    972650.96
paid       922374.20
social     976818.37
Name: gasto, dtype: float64


Con un margen de ganancia ligeramente superior al 10% se considera un buen numero para un negocio de reventa de productos, con un gasto similar en todos los canales de publicidad, se registro un tiquete promedio de 2081 con una cantidad de 7 items, lo que demuestra el gran volumen en cantidad.

---

## Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.

In [12]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [13]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [14]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios
FROM events
GROUP BY nombre_evento
ORDER BY usuarios DESC;
'''

totals = pd.read_sql(query_totals, con=engine)
totals

,nombre_evento,usuarios
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [15]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH cte_first_visit AS(
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento='first_visit'
),
cte_add_to_cart AS(
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento='add_to_cart'
),
cte_select_item AS(
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento='select_item'
),
cte_begin_checkout AS(
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento='begin_checkout'
),
cte_add_payment_info AS(
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento='add_payment_info'
),
cte_purchase AS(
    SELECT DISTINCT id_usuario
    FROM events
    WHERE nombre_evento='purchase'
)
SELECT 
    (SELECT COUNT(*) FROM cte_first_visit) AS first_visit_users,
    (SELECT COUNT(*) FROM cte_add_to_cart) AS add_to_cart_users,
    (SELECT COUNT(*) FROM cte_select_item) AS select_item_users,
    (SELECT COUNT(*) FROM cte_begin_checkout) AS begin_checkout_users,
    (SELECT COUNT(*) FROM cte_add_payment_info) AS add_payment_info_users,
    (SELECT COUNT(*) FROM cte_purchase) AS purchase_users,
        ROUND(((SELECT COUNT(*)FROM cte_first_visit)-(SELECT COUNT(*)FROM cte_add_to_cart))*100.0/NULLIF((SELECT COUNT(*)FROM cte_first_visit),0),2) AS dropoff_after_add_to_cart,
        ROUND(((SELECT COUNT(*)FROM cte_add_to_cart)-(SELECT COUNT(*)FROM cte_select_item))*100.0/NULLIF((SELECT COUNT(*)FROM cte_add_to_cart),0),2) AS dropoff_afterselect_item,
        ROUND(((SELECT COUNT(*)FROM cte_select_item)-(SELECT COUNT(*)FROM cte_begin_checkout))*100.0/NULLIF((SELECT COUNT(*)FROM cte_select_item),0),2) AS dropoff_afterbegin_checkout,
        ROUND(((SELECT COUNT(*)FROM cte_begin_checkout)-(SELECT COUNT(*)FROM cte_add_payment_info))*100.0/NULLIF((SELECT COUNT(*)FROM cte_begin_checkout),0),2) AS dropoff_after_add_payment_info,
        ROUND(((SELECT COUNT(*)FROM cte_add_payment_info)-(SELECT COUNT(*)FROM cte_purchase))*100.0/NULLIF((SELECT COUNT(*)FROM cte_add_payment_info),0),2) AS dropoff_after_purchase;
    
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

,first_visit_users,add_to_cart_users,select_item_users,begin_checkout_users,add_payment_info_users,purchase_users,dropoff_after_add_to_cart,dropoff_afterselect_item,dropoff_afterbegin_checkout,dropoff_after_add_payment_info,dropoff_after_purchase
0,7796,7634,7582,7208,6250,6240,2.08,0.68,4.93,13.29,0.16


Segun los resultados encontrados en el analisis del funnel de compras de la aplicacion, se podria decir que el paso del proceso con mas friccion para el usuario es agregar la informacion de pago seguido por el iniciar el check out, ya que si estos tuvieran el promedio de dropout similar a los demas pasos, el dropout en promedio de todo el proceso seria menor al 2% lo que haria que vasi toda operacion fuera exitosaa, por lo que se debe revisar mediante un estudio y prueba de que manera se puede reducir la friccion de esta parce del proceso para el usuario.

---

## Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

In [16]:
# Explorar tabla users
# =========================
query_users = '''
SELECT * 
FROM users 
;
'''
users = pd.read_sql(query_users, con=engine)
users.head(5)

,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free
3,user_3,2025-03-04,Mexico,desktop,free
4,user_4,2025-02-27,Argentina,desktop,free


In [17]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(5)

,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1
3,user_0,2025-02-26,28,0
4,user_1,2025-01-14,7,0


In [18]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH tabla_join AS (
    SELECT u.id_usuario, 
        CAST(u.fecha_registro AS DATE), 
        ua.dias_despues_registro, 
        ua.activo
    FROM users u
    LEFT JOIN user_activity ua ON u.id_usuario = ua.id_usuario
    ),
    cohortes AS (
    SELECT id_usuario,
        DATE_TRUNC('month',MIN(fecha_registro)) AS cohort_mensual,
        MAX(dias_despues_registro) AS dias_despues_registro,
        activo
    FROM tabla_join
    GROUP BY id_usuario, dias_despues_registro,activo
    ),
    retencion AS(
    SELECT cohort_mensual,
    COUNT(*) AS usuarios_iniciales,
    COUNT(CASE WHEN dias_despues_registro >=7 AND activo = 1 THEN 1 END) AS retencion_s1,
    COUNT(CASE WHEN dias_despues_registro >=14 AND activo = 1 THEN 1 END) AS retencion_s2,
    COUNT(CASE WHEN dias_despues_registro >=21 AND activo = 1 THEN 1 END) AS retencion_s3,
    COUNT(CASE WHEN dias_despues_registro >=28 AND activo = 1 THEN 1 END) AS retencion_s4,
    COUNT(CASE WHEN dias_despues_registro >=35 AND activo = 1 THEN 1 END) AS retencion_s5
    FROM cohortes
    GROUP BY cohort_mensual
    )
    SELECT 
        TO_CHAR(cohort_mensual, 'YYYY-MM') AS cohorte,
        usuarios_iniciales,
        ROUND(retencion_s1::numeric / usuarios_iniciales,3) AS semana_1,
        ROUND(retencion_s2::numeric / usuarios_iniciales,3) AS semana_2,
        ROUND(retencion_s3::numeric / usuarios_iniciales,3) AS semana_3,
        ROUND(retencion_s4::numeric / usuarios_iniciales,3) AS semana_4,
        ROUND(retencion_s5::numeric / usuarios_iniciales,3) AS semana_5
    FROM retencion
    ORDER BY cohorte
    ;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

,cohorte,usuarios_iniciales,semana_1,semana_2,semana_3,semana_4,semana_5
0,2025-01,6508,0.414,0.307,0.204,0.103,0.0
1,2025-02,5776,0.421,0.315,0.209,0.100,0.0
2,2025-03,6544,0.419,0.316,0.208,0.103,0.0
3,2025-04,6424,0.419,0.313,0.205,0.101,0.0
4,2025-05,6748,0.408,0.305,0.205,0.101,0.0


En el analisis por cohorte sobre la retencion de los usuarios en rappi, se puede ver que esta retencion es muy baja, es casi seguro que al cabo de un mes, un usuario no regrese a la aplicacion, por lo que se debe investigar si esta baja retencion se deba a fallas o una mala calificacion del servicio (lo cual se puede verificar al pedir data de satisfaccion del cliente), o si los clientes solo entran y disfrutan de los beneficios y promociones ofrecidos para nuevos clientes, por lo que valdria la pena realizar campanas de fidelizacion de clientes mediante nuevas promociones o beneficios. 

---

## Paso 5: Validar si los cambios generan impacto (test estadístico)

**Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

In [19]:
experimento = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv')
experimento['variante'].value_counts()
experimento.head()

,id_usuario,variante,convirtio,dispositivo,pais,duracion_sesion,timestamp
0,exp_user_0,tratamiento,0,mobile,Argentina,114.41,2025-03-28
1,exp_user_1,tratamiento,0,desktop,Mexico,170.03,2025-01-15
2,exp_user_2,control,1,mobile,Colombia,140.21,2025-03-18
3,exp_user_3,tratamiento,0,mobile,Colombia,151.45,2025-06-03
4,exp_user_4,tratamiento,0,desktop,Mexico,299.96,2025-01-12


In [20]:
conversiones = experimento.groupby('variante')['convirtio'].sum()
totales = experimento.groupby('variante')['convirtio'].count()
exitos = [conversiones['tratamiento'],conversiones['control']]
observaciones = [totales['tratamiento'],totales['control']]
z_stat,p_value = proportions_ztest(exitos,observaciones)
print(f'Estadistico z: {z_stat}')
print(f'Valor p: {p_value}')

Estadistico z: 0.8132782986429474
Valor p: 0.41605851639119995


In [22]:
alpha = 0.05 # (95% de confianza)
if p_value < alpha :
    print('Rechazamos la hipotesis nula: No hay diferencia o impacto en la conversion de compra de los usuarios aplicando los cambios al producto.')
else :
    print('No rechazamos la hipotesis nula: Hay diferencia significativa en la conversion de la compra de los usuarios aplicando los cambios al producto.')

No rechazamos la hipotesis nula: Hay diferencia significativa en la conversion de la compra de los usuarios aplicando los cambios al producto.


Para este paso se decicio usar el metodo de analsis z_test, ya que es el especializado para evualuar escenarios binarios, conversiones y demas. Con un p-value de 0.416 no se encuentra evidencia suficiente para afirmar que los cambios en la interfaz generaron un impacto significativo en la conversión.

---

## Paso 6: Comunicar los resultados (Dashboard en BI)

---

In [23]:
# https://public.tableau.com/shared/93J6H38RF?:display_count=n&:origin=viz_share_link
#  https://drive.google.com/drive/folders/1JBtBiQhwUDFzMQK8FE50GOHk-0lvHYMa